# Analyse Dataset VP/FP — Notebook 04b

## Objectif
Analyse exploratoire du dataset ML produit par `build_ml_dataset.py` (notebook 04, cellule 4).
Ce notebook est **purement analytique** : il ne génère pas de nouvelles données et ne modifie aucun fichier.

## Prérequis
- **Librairies** : `numpy`, `pandas`, `matplotlib`, `seaborn`
- **Données** : `data/processed/ml_dataset_balanced.csv` (produit par le notebook 04)
- **Modules** : aucun module `src/` requis

## Étapes principales
1. [Imports & chargement du dataset](#1-imports--chargement-du-dataset)
2. [Distribution VP/FP & caractéristiques](#2-distribution-vpfp--caract%C3%A9ristiques)
3. [Statistiques par patient](#3-statistiques-par-patient)
4. [Heatmap des corrélations inter-features](#4-heatmap-des-corr%C3%A9lations-inter-features)

> **Lien avec les notebooks précédents :** Analyse `ml_dataset_balanced.csv` issu du notebook 04.
> ⚠️ **Ne pas confondre** avec `radiomics_features.csv` (uniquement les 35 VP du GT, sans FP — inutilisable pour la classification).


## 1. Imports & chargement du dataset

### 1.1 Imports et chargement de `ml_dataset_balanced.csv`

Import des librairies de visualisation et chargement du dataset ML VP/FP.
Affichage des premières lignes pour vérifier la structure du fichier.

> **Entrée :** `data/processed/ml_dataset_balanced.csv`
> **Sortie :** `df` (DataFrame), `feature_cols` (liste des colonnes features).


In [ ]:
# Cellule 1 - Imports + chargement dataset
import sys, os, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path: sys.path.insert(0, root_path)

sns.set_style('whitegrid')
PROCESSED_DIR = Path.cwd().parent / 'data' / 'processed'

# Le dataset VP+FP vient de build_ml_dataset() — notebook 04, cellule 4
# (radiomics_features.csv = uniquement VP du GT, pas de FP — ne pas utiliser ici)
CSV_PATH = PROCESSED_DIR / 'ml_dataset_balanced.csv'

if not CSV_PATH.exists():
    print(f'Dataset introuvable : {CSV_PATH}')
    print('Executer cellule 4 du notebook 04_radiomics_demo.ipynb dabord.')
    raise FileNotFoundError(str(CSV_PATH))

df = pd.read_csv(CSV_PATH)
meta_cols    = ['PatientID', 'candidate_id', 'has_embolism']
feature_cols = [c for c in df.columns if c not in meta_cols]

n_vp = int(df['has_embolism'].sum())
n_fp = len(df) - n_vp
print(f'Dataset : {len(df)} candidats | VP={n_vp} ({100*n_vp/len(df):.1f}%) | FP={n_fp} ({100*n_fp/len(df):.1f}%)')
print(f'Features : {len(feature_cols)} | Patients : {df[chr(34)]PatientID[chr(34)].nunique()}')
display(df.head(3))


## 2. Distribution VP/FP & caractéristiques

### 2.1 Distribution des classes, volume et intensité des candidats

Visualisation de la distribution des classes (VP/FP), du volume des candidats et de leur intensité HU moyenne.
Permet de quantifier le déséquilibre des classes — typiquement 5–10% de VP pour 90–95% de FP.

> **Point méthodologique :** ce fort déséquilibre justifie l'utilisation de l'**AUC-PR** plutôt que l'**AUC-ROC** pour évaluer les classifieurs (notebook 05).


In [ ]:
# Cellule 2 - Distribution VP/FP + volume + intensite
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
counts = df['has_embolism'].value_counts().sort_index()
axes[0].bar(['FP (0)','VP (1)'], counts.values, color=['#e74c3c','#2ecc71'])
axes[0].set_title('Distribution VP / FP')
for i, v in enumerate(counts.values): axes[0].text(i, v+0.5, str(v), ha='center', fontweight='bold')

vol_col = 'shape_volume_mm3' if 'shape_volume_mm3' in df.columns else feature_cols[0]
for lbl,col,nm in [(1,'#2ecc71','VP'),(0,'#e74c3c','FP')]:
    axes[1].hist(df[df['has_embolism']==lbl][vol_col].dropna(), bins=30, alpha=0.6, color=col, label=nm)
axes[1].set_title(f'Volume : {vol_col}'); axes[1].legend()

int_col = 'intensity_mean' if 'intensity_mean' in df.columns else feature_cols[1]
for lbl,col,nm in [(1,'#2ecc71','VP'),(0,'#e74c3c','FP')]:
    axes[2].hist(df[df['has_embolism']==lbl][int_col].dropna(), bins=30, alpha=0.6, color=col, label=nm)
axes[2].set_title(f'Intensite : {int_col}'); axes[2].legend()

plt.suptitle('Dataset VP/FP - Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
os.makedirs('../figures/radiomics', exist_ok=True)
plt.savefig('../figures/radiomics/vp_fp_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


## 3. Statistiques par patient

### 3.1 Nombre de candidats VP/FP par patient

Calcul des statistiques de détection par patient : nombre total de candidats, VP, FP et pourcentage de VP.
Identifie les patients avec un taux de VP anormalement faible (problème de détection O2) ou élevé.


In [ ]:
# Cellule 3 - Stats par patient
per_patient = df.groupby('PatientID').agg(total=('has_embolism','count'), vp=('has_embolism','sum')).reset_index()
per_patient['fp']     = per_patient['total'] - per_patient['vp']
per_patient['pct_vp'] = (per_patient['vp'] / per_patient['total'] * 100).round(1)
print(f'VP moyen / patient : {per_patient[chr(34)]vp[chr(34)].mean():.1f}')
print(f'FP moyen / patient : {per_patient[chr(34)]fp[chr(34)].mean():.1f}')
print(f'Patients sans VP   : {(per_patient[chr(34)]vp[chr(34)]==0).sum()}')
display(per_patient.sort_values('vp', ascending=False))


## 4. Heatmap des corrélations inter-features

### 4.1 Matrice de corrélation des features radiomiques

Calcul et affichage de la matrice de corrélation (valeur absolue) entre toutes les features.
Les blocs de features fortement corrélées (rouge foncé) sont des candidates à la réduction par `correlation_threshold`.

> **Usage pratique :** ce graphique guide le paramètre `correlation_threshold` dans `RadiomicsFeatureSelector` (notebook 04, cellule 5).


In [ ]:
# Cellule 4 - Heatmap correlations features
X = df[feature_cols].fillna(0)
corr = X.corr().abs()
plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdYlGn_r', vmax=1, vmin=0,
            annot=(len(feature_cols)<=15), fmt='.2f', linewidths=0.5,
            xticklabels=feature_cols, yticklabels=feature_cols)
plt.title('Matrice de correlation (valeur absolue)')
plt.xticks(rotation=45, ha='right', fontsize=8); plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig('../figures/radiomics/feature_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

high = [(c1,c2,corr.loc[c1,c2]) for i,c1 in enumerate(feature_cols)
        for c2 in feature_cols[i+1:] if corr.loc[c1,c2]>0.9]
print(f'Paires > 0.9 : {len(high)}')
for c1,c2,v in sorted(high,key=lambda x:-x[2])[:10]: print(f'  {c1:30s} <-> {c2:30s}  r={v:.3f}')


### Notes

- Notebook **analytique** uniquement — ne genere pas de nouvelles donnees
- Source : `ml_dataset_balanced.csv` (VP du GT + FP de O2)
- **Ne pas confondre** avec `radiomics_features.csv` (uniquement VP du GT, sans FP)
- La heatmap guide le `correlation_threshold` dans `RadiomicsFeatureSelector`  
- Prochaine etape : `05_classical_classifier_demo.ipynb`